# Polyphonic AMT: evaluation run

Runtime > Change runtime type > **GPU (T4)** before running. Results are written to Google Drive, so a disconnect only costs the clip currently being processed: re-run the last cell and it resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/polyphonic_amt_results'
os.makedirs(OUT, exist_ok=True)

## 1. Get the code
Clones the repo, so this notebook file is the only thing you need to upload.

In [ ]:
REPO_URL = 'https://github.com/AbhishMB/Sound-Script.git'
BRANCH = 'polyphonic-amt'

%cd /content
!rm -rf Sound-Script && git clone -q -b {BRANCH} {REPO_URL} Sound-Script
%cd /content/Sound-Script/polyphonic_amt

In [ ]:
!pip install -q demucs torchcrepe mir_eval pretty_midi imageio-ffmpeg pytest

## 2. Datasets (about 2.7 GB download from Zenodo)

In [ ]:
!python evaluation/get_data.py --root /content/data
!ls /content/data/raw

## 3. Unit tests (including the Demucs / torchcrepe tests on GPU)

In [ ]:
!AMT_TEST_DEVICE=cuda python -m pytest tests -q -p no:cacheprovider

## 4. Smoke run (about 3 minutes): 3 clips from the dev singers, two noise levels

In [ ]:
!python evaluation/run_eval.py --data /content/data --split dev --per-singer 1 --limit 3 --snr 10 0 --out /content/smoke.jsonl
import pandas as pd
df = pd.read_json('/content/smoke.jsonl', lines=True)
print((df.groupby(['condition','system'])[['rpa','rpa_gated','precision','vfa']].mean()*100).round(1).loc[(slice(None), ['crepeF:raw','sepF:conf+mask','crepe:raw','sep:conf+mask']), :])

## 5. Full evaluation
Three stages (VocalSet noise sweep, vocadito noise sweep, vocadito + MUSDB accompaniment). Expect roughly 1 to 2 hours on a T4. Safe to re-run after a disconnect.

In [ ]:
!python evaluation/run_all.py --data /content/data --out {OUT}

In [ ]:
print(open(f'{OUT}/summary.md').read())